# SCAG Demo

Step-by-step demonstration of the Semantic Coactivation Graph pipeline on M&Ms-2 LAX-4C.

The notebook follows the same sequence as the refactored code:

**data → model → activations → neuron–concept matrix X → coactivation matrix R → semantic matrix Q → R–Q alignment → graph/community analysis → causal ablation**

This demo uses one architecture at a time so that every intermediate representation remains visible.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
from collections import Counter

from scag.config import SCAGConfig
from scag.data import build_graph_analysis_cohort, make_split_safe, build_loaders, phase_encode_segments
from scag.models import build_model
from scag.activations import extract_activations
from scag.semantics import compute_neuron_concept_matrix, make_semantic_views
from scag.matrices import compute_R, compute_Q, align_R_Q, compute_edge_reliability
from scag.graph import analyze_graph, density_threshold, seed_stability, community_cards, area_coupling
from scag.ablation import community_ablation
from scag.pipeline import compute_area_profiles, run_semantic_granularity

## 2. Existing project setup

The original V4 runner explicitly assumes that the following objects already exist from your current **Block 0**:

- `MNMS2SemanticDataset`
- `EDES2DClassifier`
- `MODEL_ROOT`
- `SPEC`
- `splits`
- `train_dataset`
- `val_dataset`
- `eval_transform`

Keep your existing Block-0 code here, or import those objects from the module where you move that code during the refactor.

No replacement dataset/model definitions are introduced in this demo.

In [ ]:
# Example once your existing Block-0 code has been moved into a module:
#
# from project_setup import (
#     MNMS2SemanticDataset,
#     EDES2DClassifier,
#     MODEL_ROOT,
#     SPEC,
#     splits,
#     train_dataset,
#     val_dataset,
#     eval_transform,
# )

required = [
    "MNMS2SemanticDataset", "EDES2DClassifier", "MODEL_ROOT", "SPEC",
    "splits", "train_dataset", "val_dataset", "eval_transform"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run/import your existing Block-0 setup first. Missing: " + ", ".join(missing)
    )

## 3. Configuration

In [ ]:
cfg = SCAGConfig()

print("Device:", cfg.device)
print("alpha:", cfg.alpha)
print("eta:", cfg.eta)
print("tau:", cfg.tau)
print("graph density:", cfg.graph_density)
print("concepts:", cfg.concept_name_map)

## 4. Build the graph-analysis cohort

In [ ]:
val_dataset_v4 = build_graph_analysis_cohort(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    dataset_class=MNMS2SemanticDataset,
    splits=splits,
    eval_transform=eval_transform,
)

val_dataset_safe = make_split_safe(train_dataset, val_dataset_v4)

print("Train patients:", len(train_dataset))
print("Graph-evaluation patients:", len(val_dataset_safe))

## 5. Build data loaders

In [ ]:
train_loader, val_loader = build_loaders(
    train_dataset,
    val_dataset_safe,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
)

print("Train batches:", len(train_loader))
print("Evaluation batches:", len(val_loader))

## 6. Select and load a model

The demo uses `resnet18`, which is already one of the architectures in the V4 experiment. Change `tag` to another configured architecture when required.

In [ ]:
tag = "resnet18"

model, hook_target, layer_name = build_model(
    tag=tag,
    model_root=MODEL_ROOT,
    model_class=EDES2DClassifier,
    classes=SPEC.classes,
    cfg=cfg,
)

model = model.to(cfg.device).eval()

print("Model:", tag)
print("Feature layer:", layer_name)

## 7. Extract train activations

In [ ]:
train_acts, train_segs = extract_activations(
    model, train_loader, hook_target, cfg.device
)

print("Train activations:", train_acts.shape)
print("Train masks:", train_segs.shape)

## 8. Extract evaluation activations

In [ ]:
val_acts, val_segs = extract_activations(
    model, val_loader, hook_target, cfg.device
)

print("Evaluation activations:", val_acts.shape)
print("Evaluation masks:", val_segs.shape)

## 9. Convert masks to phase-aware concepts

In [ ]:
train_segs = phase_encode_segments(train_segs)
val_segs = phase_encode_segments(val_segs)

print("Phase-aware labels:", torch.unique(train_segs).tolist())

## 10. Compute the neuron–concept association matrix X

Rows are neurons/filters and columns are:

`background, RV_ED, MYO_ED, LV_ED, RV_ES, MYO_ES, LV_ES`.

In [ ]:
X, labeled = compute_neuron_concept_matrix(
    train_acts,
    train_segs,
    num_concepts=cfg.num_concepts,
    alpha=cfg.alpha,
    eta=cfg.eta,
)

lab_idx = np.where(labeled)[0]

print("X shape:", X.shape)
print("Labeled neurons:", len(lab_idx), "/", X.shape[0])

## 11. Inspect dominant concepts

In [ ]:
label_counts = Counter(X[lab_idx].argmax(axis=1).tolist())
readable_counts = {
    cfg.concept_name_map[int(k)]: int(v)
    for k, v in sorted(label_counts.items())
}

readable_counts

## 12. Compute the coactivation matrix R

Each activation map is spatially pooled and Spearman correlation is computed between every pair of neurons.

In [ ]:
R_train = compute_R(train_acts)
R_eval = compute_R(val_acts)

print("R_train:", R_train.shape)
print("R_eval:", R_eval.shape)

## 13. Compute the semantic similarity matrix Q

`Q` is cosine similarity between the neuron concept profiles in `X`.  
For alignment, only the neurons selected by the original phase-aware dissection are used.

In [ ]:
Q = compute_Q(X[lab_idx])

print("Q shape:", Q.shape)
print("Expected:", (len(lab_idx), len(lab_idx)))

## 14. Train–evaluation edge reliability

In [ ]:
edge_reliability = compute_edge_reliability(
    R_train, R_eval, lab_idx
)

print(f"Edge reliability: rho={edge_reliability:.4f}")

## 15. Align R and Q

In [ ]:
R_train_lab = R_train[np.ix_(lab_idx, lab_idx)]
R_eval_lab = R_eval[np.ix_(lab_idx, lab_idx)]

alignment_train = align_R_Q(
    R_train_lab,
    Q,
    n_perm=cfg.n_mantel_permutations,
)

alignment_eval = align_R_Q(
    R_eval_lab,
    Q,
    n_perm=cfg.n_mantel_permutations,
)

print("Train alignment (r, z, p):", alignment_train)
print("Eval alignment  (r, z, p):", alignment_eval)

## 16. Build and analyze the fixed-threshold coactivation graph

In [ ]:
graph_eval, artifacts_eval = analyze_graph(
    R_eval,
    X,
    labeled,
    tau=cfg.tau,
    n_perm=cfg.n_battery_permutations,
)

graph_eval

## 17. Inspect Louvain communities

In [ ]:
communities = artifacts_eval["communities"]

print("Number of communities:", len(communities))
for i, community in enumerate(communities):
    print(f"Community {i:02d}: {len(community)} neurons")

## 18. Density-matched graph

In [ ]:
tau_dm = density_threshold(R_eval, cfg.graph_density)

graph_dm, artifacts_dm = analyze_graph(
    R_eval,
    X,
    labeled,
    tau=tau_dm,
    n_perm=cfg.n_battery_permutations,
)

print("Density-matched threshold:", tau_dm)
print("Edges:", graph_dm["n_edges"])
print("Modularity:", graph_dm["modularity"])

## 19. Compare anatomy, phase, and anatomy × phase semantics

In [ ]:
granularity_df, granularity_summary, semantic_views = run_semantic_granularity(
    tag=tag,
    R_train=R_train,
    R_eval=R_eval,
    X=X,
    labeled=labeled,
    tau_dm=tau_dm,
    cfg=cfg,
)

granularity_df

## 20. Louvain seed stability

In [ ]:
ami_mean, ami_sd, partition_stability = seed_stability(
    artifacts_eval["G"],
    artifacts_eval["lab_nodes"],
    artifacts_eval["labels"],
    n_seeds=cfg.n_louvain_seeds,
)

print("AMI mean:", ami_mean)
print("AMI SD:", ami_sd)
print("Partition stability:", partition_stability)

## 21. Community semantic enrichment

In [ ]:
enrichment_df = community_cards(
    artifacts_eval,
    cfg.concept_name_map,
    cfg.concept_category_map,
)

enrichment_df

## 22. Anatomy-area coupling

In [ ]:
area_profiles = compute_area_profiles(train_segs, cfg.num_concepts)

area_stats = area_coupling(
    R_eval,
    X,
    labeled,
    area_profiles,
    n_perm=cfg.n_cooccurrence_permutations,
)

area_stats

## 23. Causal community ablation

Each detected community is ablated by zeroing its channels at the selected feature layer.  
The prediction change is measured with KL divergence and compared with size-matched random neuron ablations.

In [ ]:
node2label = dict(zip(
    artifacts_eval["lab_nodes"],
    artifacts_eval["labels"].tolist(),
))

ablation_df = community_ablation(
    model=model,
    hook_target=hook_target,
    loader=val_loader,
    communities=artifacts_eval["communities"],
    num_filters=R_eval.shape[0],
    device=cfg.device,
    val_segs=val_segs,
    node2label=node2label,
    concept_names=cfg.concept_name_map,
    n_random=cfg.n_ablation_samples,
)

ablation_df

## 24. Inspect community importance

In [ ]:
if ablation_df is not None and len(ablation_df):
    display(
        ablation_df[
            ["comm", "size", "kl", "rnd", "rnd_sd", "flip"]
        ].sort_values("kl", ascending=False)
    )

## End-to-end representations

At this point the demo has exposed the complete SCAG chain:

- `train_acts`, `val_acts`: activation maps
- `X`: neuron–concept association matrix
- `R_train`, `R_eval`: functional/coactivation matrices
- `Q`: semantic neuron–neuron similarity
- `alignment_*`: R–Q alignment
- `artifacts_eval["G"]`: coactivation graph
- `artifacts_eval["communities"]`: Louvain communities
- `enrichment_df`: community semantics
- `ablation_df`: causal community importance

The production runner can call `run_model()` to execute the same pipeline automatically; this notebook intentionally shows every stage individually.